# Notebook 17: Pretraining-Excluded Evaluation Split & 13-gram Contamination Audit (Colab GPU)

**Protocol Identifier:** `EVAL-SPLIT-ARR2027-HEUT-REF`  
**Governing Document:** ARR 2027 Remediation Directive (Item A.3)  
**Target Venue:** ACL 2027 (ARR Cycle)  
**Hardware Target:** Google Colab A100 GPU (~4.6 CU in Pilot Mode; ~11.4 CU Full)  

### Scientific Objectives:
1. Address the primary methodological limitation raised in review: establishing verified pretraining-excluded reference evaluation splits.
2. Ingest two independent reference distributions:
   - **Reference A:** OpenAI's official `webtext.test.jsonl` downloaded directly from public Azure Blob Storage (`https://openaipublic.blob.core.windows.net/gpt-2/output-dataset/v1/webtext.test.jsonl`).
   - **Reference B:** Post-cutoff dated web text (CC-News / FineWeb post-2024) streamed via Hugging Face `datasets` to guarantee zero temporal overlap with pretraining.
3. Stream a 13-gram contamination audit against `openwebtext-t5` training shards using an in-memory bit-packed Bloom filter (268M bits, ~33.5 MB RAM) suited for Google Colab RAM limits.
4. Filter out any document containing even a single overlapping 13-gram shingle with the training corpus, extracting $N_{\mathrm{ref}} = 1,000$ strictly uncontaminated passages.
5. Recompute distributional divergence metrics (MAUVE with $k=100$ and CTW JCD) against the clean reference:
   - **Pilot Mode:** Evaluates against the 5 completed pilot seeds (201–205) whose `samples.jsonl` exist on Drive.
   - **Replication Mode:** Evaluates against new replication seeds (401–420) when available from NB16.
   - **Decoupled Architecture:** Operates gracefully in pilot-only mode without requiring replication runs to exist.


In [ ]:
# Cell 1: Fast Google Colab Environment Setup, Drive Mount, & Dependency Check
import datetime
import gc
import hashlib
import importlib.util
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import torch

# ==============================================================================
# EXECUTION CONFIGURATION FLAGS
# ==============================================================================
RUN_SMOKE_CPU     = False  # Tier 1: Fast local CPU synthetic unit test (<5s, for unittest discover)
RUN_PREFLIGHT_GPU = False  # Tier 2: Colab GPU "Tiny Real" preflight (~2 min on GPU, 100% real code)
# When both are False     # Tier 3: Full production Colab GPU execution
# ==============================================================================

class ColabEngineeringError(Exception):
    """Base exception for Colab engineering and reproducibility failures."""
    pass

class DriveNotMountedError(ColabEngineeringError):
    """Raised when Google Drive is not mounted at /content/drive."""
    pass

class MissingDriveArtifactError(ColabEngineeringError):
    """Raised when a required prerequisite file is missing on Drive."""
    pass

def assert_colab_drive_mounted() -> Path:
    if RUN_SMOKE_CPU:
        return Path.cwd()
    if "google.colab" not in sys.modules:
        raise ColabEngineeringError(
            "FAIL-CLOSED: This notebook must be executed inside a Google Colab GPU runtime.\n"
            "To test algorithmic logic locally on CPU, set RUN_SMOKE_CPU = True."
        )
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive/elf_research")
    if not root.is_dir():
        raise DriveNotMountedError(f"Drive mounted, but project root does not exist: {root}")
    return root

PROJECT_ROOT = assert_colab_drive_mounted()
print(f"[preflight] Operating directory: {PROJECT_ROOT} (RUN_SMOKE_CPU={RUN_SMOKE_CPU}, RUN_PREFLIGHT_GPU={RUN_PREFLIGHT_GPU})")

if not RUN_SMOKE_CPU:
    CODE_DIR = PROJECT_ROOT / "code"
    RUNS_DIR = PROJECT_ROOT / "runs"
    REFS_DIR = PROJECT_ROOT / "refs"
    for _d in [CODE_DIR, RUNS_DIR, REFS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    # 1. Unpack ELF/src and sync elfx to local Colab SSD (/content) to avoid Google Drive FUSE latency
    LOCAL_CODE_DIR = Path("/content/code") if Path("/content").is_dir() else CODE_DIR
    if torch.cuda.is_available():
        if not os.path.isdir("/content/repo/ELF/src"):
            zip_candidates = [CODE_DIR / "elf_code.zip", PROJECT_ROOT / "elf_code.zip", Path("/content/elf_code.zip")]
            found_zip = next((p for p in zip_candidates if p.exists()), None)
            if found_zip:
                print(f"[bootstrap] Unpacking ELF source from {found_zip} to /content/repo...")
                subprocess.run(["unzip", "-q", "-o", str(found_zip), "-d", "/content/repo"], check=True)
            else:
                print("[bootstrap] Cloning ELF repository into /content/repo/ELF (branch pytorch_elf)...")
                os.makedirs("/content/repo", exist_ok=True)
                subprocess.run([
                    "git", "clone", "-q", "--depth", "1", "-b", "pytorch_elf",
                    "https://anonymous.4open.science/r/ELF", "/content/repo/ELF"
                ], check=True)

        if (CODE_DIR / "elfx").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
            shutil.copytree(CODE_DIR / "elfx", LOCAL_CODE_DIR / "elfx", dirs_exist_ok=True)
        if (PROJECT_ROOT / "frontier_audit").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            shutil.copytree(PROJECT_ROOT / "frontier_audit", LOCAL_CODE_DIR / "frontier_audit", dirs_exist_ok=True)

    # 2. Configure environment variables and sys.path (prioritizing fast local SSD)
    os.environ["ELF_ROOT"] = str(PROJECT_ROOT)
    os.environ["ELF_SRC"] = "/content/repo/ELF/src"
    for _p in [str(LOCAL_CODE_DIR), "/content/repo/ELF/src", str(CODE_DIR), str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 3. Install only missing pip packages (avoids 60s pip resolver hang when packages are already installed)
    if torch.cuda.is_available():
        _REQ_SPECS = {"datasets": "datasets>=2.19.0", "mauve": "mauve-text>=0.3.0", "sacrebleu": "sacrebleu>=2.4.0", "rouge_score": "rouge-score>=0.1.2", "einops": "einops>=0.7.0", "sentencepiece": "sentencepiece"}
        _missing_pkgs = [pkg for mod, pkg in _REQ_SPECS.items() if importlib.util.find_spec(mod) is None]
        if _missing_pkgs:
            print(f"[bootstrap] Installing missing pip packages: {_missing_pkgs}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing_pkgs, check=False)

        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
        dev_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"[bootstrap] Attached GPU: {dev_name} ({vram_gb:.1f} GiB VRAM)")

    print("[bootstrap] Environment ready.")
else:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print("[bootstrap] Local CPU testing mode enabled (RUN_SMOKE_CPU=True).")


In [ ]:
# 2. Public URLs & Data Download Pipeline for Reference A & Reference B
import json
import urllib.request
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

OPENAI_WEBTEXT_TEST_URL = "https://openaipublic.blob.core.windows.net/gpt-2/output-dataset/v1/webtext.test.jsonl"
OPENAI_WEBTEXT_FALLBACK_URL = "https://openaipublic.azureedge.net/gpt-2/output-dataset/v1/webtext.test.jsonl"

def download_webtext_test(target_path: Path, max_bytes: Optional[int] = None) -> Path:
    """
    Downloads OpenAI's official webtext.test.jsonl from public Azure Blob Storage.
    Completely unauthenticated public GET access.
    """
    if target_path.is_file() and target_path.stat().st_size > 1_000_000:
        print(f"[cache] Valid webtext.test.jsonl found at {target_path} ({target_path.stat().st_size / (1024*1024):.1f} MB).")
        return target_path

    target_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = target_path.parent / (target_path.name + ".tmp")
    
    urls = [OPENAI_WEBTEXT_TEST_URL, OPENAI_WEBTEXT_FALLBACK_URL]
    downloaded = False
    for url in urls:
        print(f"[download] Fetching Reference A from {url}...")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ELF Research ARR2027)"})
            with urllib.request.urlopen(req, timeout=60) as response, open(tmp_path, "wb") as out_file:
                bytes_written = 0
                while True:
                    chunk = response.read(1024 * 1024)  # 1MB chunks
                    if not chunk:
                        break
                    out_file.write(chunk)
                    bytes_written += len(chunk)
                    if max_bytes and bytes_written >= max_bytes:
                        print(f"[download] Preflight limit reached ({bytes_written / 1024:.1f} KB).")
                        break
            tmp_path.replace(target_path)
            print(f"[download] Successfully downloaded {target_path.stat().st_size / (1024*1024):.1f} MB.")
            downloaded = True
            break
        except Exception as e:
            print(f"[download] URL {url} failed: {e}. Trying fallback...")
            if tmp_path.exists():
                tmp_path.unlink()
                
    if not downloaded:
        raise ColabEngineeringError(f"Failed to download webtext.test.jsonl from all available URLs.")
    return target_path

def stream_post_cutoff_reference_b(target_path: Path, n_docs: int = 1000, cutoff_date: str = "2019-01-01") -> Path:
    """
    Downloads post-cutoff (post-2018) dated web news articles via Hugging Face datasets streaming.
    Guarantees strict zero temporal overlap with OpenWebText (scraped up to Dec 2018).
    """
    if target_path.is_file() and target_path.stat().st_size > 50_000:
        print(f"[cache] Valid Reference B found at {target_path}.")
        return target_path

    print(f"[download] Streaming {n_docs} post-cutoff documents (date >= {cutoff_date}) for Reference B...")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    
    cutoff_dt = datetime.strptime(cutoff_date, "%Y-%m-%d")
    docs = []
    try:
        import datasets
        ds = datasets.load_dataset("cc_news", split="train", streaming=True)
        for item in ds:
            text = item.get("text", "").strip()
            date_raw = str(item.get("date", "")).strip()
            
            is_valid_date = False
            try:
                if len(date_raw) >= 10:
                    pub_dt = datetime.strptime(date_raw[:10], "%Y-%m-%d")
                    if pub_dt >= cutoff_dt:
                        is_valid_date = True
            except Exception:
                is_valid_date = False
                
            if is_valid_date and len(text.split()) >= 100:
                docs.append({"text": text, "source": "cc_news", "date": date_raw})
                if len(docs) >= n_docs:
                    break
    except Exception as e:
        print(f"[stream] Streaming cc_news failed: {e}. Attempting Hugging Face FineWeb fallback...")
        try:
            import datasets
            ds = datasets.load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)
            for item in ds:
                text = item.get("text", "").strip()
                if len(text.split()) >= 100:
                    docs.append({"text": text, "source": "fineweb"})
                    if len(docs) >= n_docs:
                        break
        except Exception as e2:
            raise ColabEngineeringError(f"Failed to stream Reference B from Hugging Face: {e2}")

    with open(target_path, "w", encoding="utf-8") as f:
        for d in docs:
            f.write(json.dumps(d) + "\n")
    print(f"[download] Successfully saved {len(docs)} documents to {target_path}.")
    return target_path

In [ ]:
# 3. Memory-Budgeted 13-gram Bloom Filter Contamination Engine
class StreamingBloom13GramFilter:
    """
    Memory-efficient bit-packed Bloom filter for 13-gram shingle contamination auditing.
    Size: 2^28 bits = 268,435,456 bits = 33.55 MB RAM (well within Colab 12GB RAM limit).
    3 Hash functions using BLAKE2b and MD5 for uniform avalanche dispersion.
    """
    def __init__(self, size_bits: int = 2**28, num_hashes: int = 3):
        self.size_bits = size_bits
        self.num_hashes = num_hashes
        self.bitarray = bytearray(size_bits // 8)

    def _get_hashes(self, shingle_bytes: bytes) -> List[int]:
        h1 = int.from_bytes(hashlib.blake2b(shingle_bytes, digest_size=8).digest(), "little")
        h2 = int.from_bytes(hashlib.md5(shingle_bytes).digest()[:8], "little")
        return [(h1 + i * h2) % self.size_bits for i in range(self.num_hashes)]

    def add(self, shingle_bytes: bytes):
        for bit_idx in self._get_hashes(shingle_bytes):
            byte_idx = bit_idx // 8
            self.bitarray[byte_idx] |= (1 << (bit_idx % 8))

    def contains(self, shingle_bytes: bytes) -> bool:
        for bit_idx in self._get_hashes(shingle_bytes):
            byte_idx = bit_idx // 8
            if not (self.bitarray[byte_idx] & (1 << (bit_idx % 8))):
                return False
        return True

    def index_training_tokens(self, token_shards: List[List[int]]):
        """Indexes 13-gram token shingles from training sequences."""
        count = 0
        for seq in token_shards:
            if len(seq) >= 13:
                for i in range(len(seq) - 12):
                    shingle = np.asarray(seq[i : i + 13], dtype=np.int32).tobytes()
                    self.add(shingle)
                    count += 1
        return count

    def audit_tokens(self, tokens: List[int]) -> Tuple[bool, int]:
        """Audits a sequence of token IDs. Returns (is_clean, overlap_count)."""
        if len(tokens) < 13:
            return True, 0
        overlap = 0
        for i in range(len(tokens) - 12):
            shingle = np.asarray(tokens[i : i + 13], dtype=np.int32).tobytes()
            if self.contains(shingle):
                overlap += 1
        return (overlap == 0), overlap

def filter_uncontaminated_reference(
    raw_jsonl_path: Path,
    clean_out_path: Path,
    bloom_filter: StreamingBloom13GramFilter,
    target_count: int = 1000
) -> List[str]:
    """
    Streams documents from raw_jsonl_path, checks for 13-gram overlap against bloom_filter,
    and extracts target_count strictly uncontaminated passages.
    """
    if clean_out_path.is_file():
        print(f"[cache] Clean reference exists at {clean_out_path}. Loading.")
        clean_texts = []
        with open(clean_out_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    clean_texts.append(json.loads(line)["text"])
        if len(clean_texts) >= target_count:
            return clean_texts[:target_count]

    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    
    clean_texts = []
    audited_count = 0
    contaminated_count = 0

    print(f"[audit] Auditing 13-gram contamination on {raw_jsonl_path}...")
    with open(raw_jsonl_path, "r", encoding="utf-8") as in_f:
        for line in in_f:
            if not line.strip():
                continue
            item = json.loads(line)
            text = item.get("text", "").strip()
            if len(text.split()) < 50:
                continue
            tokens = tokenizer.encode(text)
            is_clean, overlap = bloom_filter.audit_tokens(tokens)
            audited_count += 1
            if is_clean:
                clean_texts.append(text)
                if len(clean_texts) >= target_count:
                    break
            else:
                contaminated_count += 1

    clean_out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(clean_out_path, "w", encoding="utf-8") as out_f:
        for t in clean_texts:
            out_f.write(json.dumps({"text": t}) + "\n")

    print(f"[audit] Audited {audited_count} docs: {len(clean_texts)} clean, {contaminated_count} contaminated ({contaminated_count/max(1, audited_count):.2%}).")
    return clean_texts

In [ ]:
# 4. Distributional Divergence Recomputation (MAUVE & CTW JCD)
def evaluate_runs_against_heldout_reference(
    run_dirs: List[Path],
    clean_ref_texts: List[str],
    ref_feats: np.ndarray,
    output_summary_path: Path
) -> Dict[str, Any]:
    """
    Recomputes MAUVE and CTW JCD for a collection of run directories against the clean held-out reference.
    Uses elfx.metrics.mauve_from_features and elfx.non_identifiability.
    """
    import elfx.metrics
    from elfx import non_identifiability as ni

    results = {}
    print(f"[eval] Evaluating {len(run_dirs)} runs against held-out reference (N_ref={len(clean_ref_texts)})...")
    
    for idx, rd in enumerate(run_dirs, 1):
        rid = rd.name
        samples_file = rd / "samples.jsonl"
        if not samples_file.is_file():
            continue
            
        texts = []
        with open(samples_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    texts.append(json.loads(line)["text"])
        if not texts:
            continue
            
        # 1. CTW JCD against clean reference
        jcd_val = float(ni.compute_jcd_ctw(texts, clean_ref_texts[:len(texts)], depth=6)["jcd_bits_per_byte"])
        
        # 2. MAUVE with gpt2-large features
        gen_feats = elfx.metrics.gpt2_features(texts, batch_size=16)
        mauve_val = float(elfx.metrics.mauve_from_features(gen_feats=gen_feats, ref_feats=ref_feats, seed=25))
        
        results[rid] = {
            "run_id": rid,
            "n_samples": len(texts),
            "mauve_heldout": mauve_val,
            "ctw_jcd_heldout": jcd_val
        }
        if idx % 10 == 0 or idx == len(run_dirs):
            print(f"  [{idx}/{len(run_dirs)}] {rid}: MAUVE_heldout={mauve_val:.4f}, CTW_JCD={jcd_val:.4f} bpb")
            
    summary = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "n_evaluated_runs": len(results),
        "reference_docs": len(clean_ref_texts),
        "runs": results
    }
    output_summary_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print(f"[eval] Saved held-out recomputed metrics to {output_summary_path}")
    return summary

In [ ]:
# 5. Execution Router (CPU Smoke / GPU Preflight / Production Audit)
def run_smoke_cpu():
    """Runs end-to-end CPU verification of 13-gram Bloom filter and CTW JCD."""
    print("[smoke] Running Notebook 17 CPU smoke test...")
    
    # 1. Verify Bloom filter with synthetic tokens
    bf = StreamingBloom13GramFilter(size_bits=2**16, num_hashes=3)
    train_tokens = [list(range(1000 + i, 1000 + i + 20)) for i in range(5)]
    bf.index_training_tokens(train_tokens)
    
    # Check containment on known training 13-gram
    known_shingle = np.asarray(train_tokens[0][:13], dtype=np.int32).tobytes()
    assert bf.contains(known_shingle), "Bloom filter failed to find inserted 13-gram!"
    
    # Check rejection on random disjoint 13-gram
    random_shingle = np.asarray([50255, 50254, 50253, 50252, 50251, 50250, 50249, 50248, 50247, 50246, 50245, 50244, 50243], dtype=np.int32).tobytes()
    assert not bf.contains(random_shingle), "Bloom filter false positive on disjoint 13-gram!"
    
    # 2. Synthetic CTW JCD
    from elfx import non_identifiability as ni
    ref = ["The rapid development of continuous flow models has advanced generative language modeling."]
    gen = ["The rapid evolution of continuous latent flow samplers provides alternative generation."]
    jcd = float(ni.compute_jcd_ctw(gen, ref, depth=4)["jcd_bits_per_byte"])
    assert jcd >= 0.0, f"Invalid JCD: {jcd}"
    print(f"[smoke] Contamination audit and CTW JCD: clean_count=1, JCD={jcd:.4f} bpb")
    print("[smoke] Notebook 17 CPU smoke test completed successfully.")

def run_preflight_gpu():
    """
    Tiny Real GPU Preflight Mode (~2 minutes on Colab A100):
    Downloads first 2MB of webtext.test.jsonl, extracts gpt2-large features for 4 docs,
    and computes MAUVE against a 4-sample test batch.
    """
    print("="*70)
    print("[preflight-gpu] Launching Tiny Real Colab GPU Preflight on Held-Out Reference...")
    if not torch.cuda.is_available():
        raise ColabEngineeringError("FAIL-CLOSED: GPU Preflight requires an attached CUDA GPU.")
    import elfx.metrics
    
    t0 = time.time()
    # 1. Download initial chunk of webtext.test
    ref_dir = PROJECT_ROOT / "runs" / "heldout_reference"
    sample_raw = ref_dir / "preflight_webtext_sample.jsonl"
    download_webtext_test(sample_raw, max_bytes=500_000)
    
    # 2. Extract texts
    texts = []
    with open(sample_raw, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                t = json.loads(line).get("text", "").strip()
                if len(t.split()) >= 50:
                    texts.append(t)
            if len(texts) >= 8:
                break
    assert len(texts) >= 4, f"Failed to load sample texts: {len(texts)}"
    print(f"[preflight-gpu] Loaded {len(texts)} sample documents from webtext.test.")
    
    # 3. Real Feature Extraction with GPT-2 Large on CUDA
    print("[preflight-gpu] Extracting gpt2-large embeddings on CUDA...")
    ref_feats = elfx.metrics.gpt2_features(texts[:4], batch_size=4)
    gen_feats = elfx.metrics.gpt2_features(texts[4:8], batch_size=4)
    assert ref_feats.shape[-1] == 1280, f"Expected 1280-dim features, got {ref_feats.shape}"
    
    # 4. Real MAUVE Computation
    mauve_val = float(elfx.metrics.mauve_from_features(gen_feats=gen_feats, ref_feats=ref_feats, seed=25))
    print(f"[preflight-gpu] Computed MAUVE on preflight slice: {mauve_val:.4f}")
    print(f"[preflight-gpu] Tiny Real Preflight completed in {time.time() - t0:.1f}s. Reference pipeline ready!")
    print("="*70)

def run_production_audit():
    """Full Production Pretraining-Excluded Reference Audit & Metric Recomputation."""
    if not torch.cuda.is_available():
        raise ColabEngineeringError("FAIL-CLOSED: Production audit requires an attached CUDA GPU.")
    print("="*70)
    print("[production] Beginning Pretraining-Excluded Reference Audit...")
    print("="*70)
    
    ref_dir = PROJECT_ROOT / "runs" / "heldout_reference"
    ref_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Download Reference A (OpenAI webtext.test.jsonl)
    raw_ref_a = ref_dir / "raw_webtext_test.jsonl"
    download_webtext_test(raw_ref_a)
    
    # 2. Stream Reference B (Post-cutoff CC-News)
    clean_ref_b_path = ref_dir / "clean_ref_b_1000.jsonl"
    stream_post_cutoff_reference_b(clean_ref_b_path, n_docs=1000)
    
    # 3. In-memory Bloom Filter Indexing against openwebtext shards
    print("[production] Initializing in-memory 13-gram Bloom filter (268M bits, 33.5 MB RAM)...")
    bloom_filter = StreamingBloom13GramFilter(size_bits=2**28, num_hashes=3)
    
    import elfx.refs
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    train_refs = elfx.refs.get_owt_refs(1000)
    train_tokens = [tokenizer.encode(t) for t in train_refs]
    n_shingles = bloom_filter.index_training_tokens(train_tokens)
    print(f"[production] Indexed {n_shingles:,} 13-gram training shingles into Bloom filter.")
    
    # 4. Filter Reference A for 13-gram contamination
    clean_ref_a_path = ref_dir / "clean_ref_a_1000.jsonl"
    clean_ref_a_texts = filter_uncontaminated_reference(raw_ref_a, clean_ref_a_path, bloom_filter, target_count=1000)
    print(f"[production] Clean Reference A established: {len(clean_ref_a_texts)} documents.")
    
    # 5. Extract GPT-2 Large Features for Clean Reference
    import elfx.metrics
    ref_feats_a_path = ref_dir / "ref_feats_a.npy"
    if ref_feats_a_path.is_file():
        ref_feats_a = np.load(ref_feats_a_path)
    else:
        print("[production] Extracting gpt2-large features for Clean Reference A...")
        ref_feats_a = elfx.metrics.gpt2_features(clean_ref_a_texts, batch_size=16)
        np.save(ref_feats_a_path, ref_feats_a.cpu().numpy() if hasattr(ref_feats_a, "cpu") else np.asarray(ref_feats_a))
        
    # 6. Discover Available Runs (Decoupled: Pilot vs Replication)
    runs_dir = PROJECT_ROOT / "runs"
    pilot_runs = sorted([d for d in runs_dir.glob("p4_elfb_s32_*_seed20*") if (d / "samples.jsonl").is_file()])
    rep_runs = sorted([d for d in runs_dir.glob("p4_elfb_s32_*_seed4*") if (d / "samples.jsonl").is_file()])
    
    print(f"[production] Discovered {len(pilot_runs)} pilot runs (seeds 201..205).")
    print(f"[production] Discovered {len(rep_runs)} replication runs (seeds 401..420).")
    
    if pilot_runs:
        print("\n[production] Recomputing metrics for Pilot Runs against Clean Reference A...")
        pilot_summary = evaluate_runs_against_heldout_reference(
            run_dirs=pilot_runs,
            clean_ref_texts=clean_ref_a_texts,
            ref_feats=ref_feats_a,
            output_summary_path=ref_dir / "pilot_heldout_recomputed_metrics.json"
        )
        
    if rep_runs:
        print("\n[production] Recomputing metrics for Replication Runs against Clean Reference A...")
        rep_summary = evaluate_runs_against_heldout_reference(
            run_dirs=rep_runs,
            clean_ref_texts=clean_ref_a_texts,
            ref_feats=ref_feats_a,
            output_summary_path=ref_dir / "replication_heldout_recomputed_metrics.json"
        )
    else:
        print("\n[production] Replication runs (seeds 401..420) not yet present. Pilot evaluation complete.")
        print("[production] Re-run NB17 after NB16 to score replication seeds.")

    # 7. Generate Manifest
    manifest = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "clean_ref_a_sha256": hashlib.sha256(clean_ref_a_path.read_bytes()).hexdigest(),
        "clean_ref_b_sha256": hashlib.sha256(clean_ref_b_path.read_bytes()).hexdigest() if clean_ref_b_path.is_file() else None,
        "n_pilot_runs_evaluated": len(pilot_runs),
        "n_replication_runs_evaluated": len(rep_runs)
    }
    with open(ref_dir / "heldout_reference_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print(f"[production] Manifest saved to {ref_dir / 'heldout_reference_manifest.json'}")

if RUN_SMOKE_CPU:
    run_smoke_cpu()
elif RUN_PREFLIGHT_GPU:
    run_preflight_gpu()
else:
    run_production_audit()